# Gated latent compression - Kaggle runner
Settings: **Accelerator = GPU T4 x1**, **Internet = On**. Run top to bottom. Each step writes to disk, so you can resume after a crash.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Get the code: either `git clone` your GitHub repo or upload the zip as a Kaggle dataset.
!git clone https://github.com/<YOUR_USER>/<YOUR_REPO>.git /kaggle/working/glc
%cd /kaggle/working/glc
!pip install -q -r requirements.txt

## 1. Plumbing smoke test (tiny random model, ~1-3 min). Must print `SMOKE TEST PASSED`.

In [ ]:
!python tests/smoke_test.py 2>&1 | tail -15

## 2. Data, gate, sanity check

In [ ]:
!python scripts/download_data.py
!python -m src.data.prepare_cceval
!python -m src.gate.train_gate
!python -m src.sanity_check

## 3. Timing probe: 2 optimizer steps. Multiply seconds/step by the total steps to plan the run.

In [ ]:
!python -m src.training.train_compressor --max_steps 2

## 4. Train compressor + LoRA (resume with `--resume` if the session dies)

In [ ]:
!python -m src.training.train_compressor

## 5. tau sweep on validation -> Pareto plot + chosen tau, then final test evaluation

In [ ]:
!python -m src.evaluation.sweep_gate --split val
!python -m src.evaluation.run_eval --split test
!python scripts/make_report.py

In [ ]:
from IPython.display import Image, display, Markdown
display(Image('results/pareto_val.png'))
display(Markdown(open('results/RESULTS.md').read()))

## 6. Zip checkpoints + results to download

In [ ]:
!zip -qr /kaggle/working/glc_outputs.zip checkpoints results logs -x 'checkpoints/train_state.pt'
!ls -lh /kaggle/working/glc_outputs.zip